# CircuitSight — Fine-Tuning & Evaluation (Qwen2.5-VL-3B, QLoRA)

Trains a small vision-language model to **read a circuit schematic and solve it** (components → topology → equations → values), using the dataset produced by `CircuitSight_dataset_generation.ipynb`.

**Run order:**
1. Setup + GPU check
2. Load the dataset (from Drive or an uploaded zip)
3. Load Qwen2.5-VL-3B (4-bit) + LoRA
4. **Day-1 OOM smoke test — the go/no-go gate.** Run this *before* committing to a full run. If it OOMs, follow the fallback notes (lower resolution or smaller model) before proceeding.
5. Full training
6. Inference
7. **Evaluation harness** — scores base vs. tuned on: component accuracy, R_eq accuracy, final-answer accuracy, and — separately — **fabrication rate** vs. **honest-abstention rate**.

Needs a GPU runtime (Runtime → Change runtime type → GPU). With Colab credits, an **A100 40GB** is comfortable for the 3B; a T4/L4 works at lower resolution or with SmolVLM.


## 1. Setup

In [1]:
# Install a torch version Unsloth supports (let pip choose the right CUDA build).
!pip install -q "torch==2.6.0" "torchvision==0.21.0"
!pip install -q unsloth sympy
print("installed — RESTART SESSION before importing")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [2]:
import torch; print("torch", torch.__version__)
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
print("unsloth ready")

torch 2.10.0+cu128
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth ready


## 2. Config

In [8]:
CFG = dict(
    MODEL       = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit",  # fallback: SmolVLM / a 2B if VRAM is tight
    MAX_IMAGE_PX= 512,     # longest image side; the main VRAM + speed knob. 512 ~= 2x faster than 768 on a T4.
    LORA_R      = 16, LORA_ALPHA = 16,
    BATCH       = 2, GRAD_ACCUM = 4,   # effective batch = BATCH*GRAD_ACCUM = 8. Bigger BATCH = faster but more VRAM; run the §4 smoke test first. OOM? -> BATCH=1, GRAD_ACCUM=8 (same effective 8, no extra VRAM). A100: BATCH 4-8 ok.
    MAX_STEPS   = 350,     # <-- caps optimizer steps (~30-40 min on a T4). Set to None to train a full epoch.
    EPOCHS      = 1,       # only used when MAX_STEPS is None
    LR          = 2e-4,
    MAX_LEN     = 2048,
    SEED        = 3407,
    DATA_DIR    = "circuitsight_dataset",   # unzipped dataset folder
    SAVE_TO_DRIVE = False,   # if True, snapshot the trained adapter to DRIVE_DIR/models (keep 2 most recent)
    DRIVE_DIR   = "/content/drive/MyDrive/CircuitSight",   # your project folder in Google Drive
    OUT_DIR     = "circuitsight_qlora",
)
# Why this many steps? EPOCHS=1 over ~10k images at effective batch BATCH*GRAD_ACCUM=4 is ~2,600
# optimizer steps (~6h on a T4) -- that is steps, not epochs. This narrow, structured behavior is
# learned in a few hundred steps, so we cap with MAX_STEPS. 350 steps ~= 2,800 images seen (effective batch 8); the
# trainer shuffles the full dataset, so all families + value-modes (numeric/symbolic/mixed) are
# sampled in proportion to the dataset mix (see CONFIG fractions). Watch the section-9 eval curve;
# raise MAX_STEPS (or set None) only if it's still climbing.
INSTRUCTION = ("You are a circuit analysis tutor. Look at the schematic and answer the question. "
    "First list every component, each with the image region it occupies as "
    "<box>[x0,y0,x1,y1]</box> in 0-1000 normalized coordinates. Then state the concepts used and "
    "the topology (what is in series/parallel). If there is a capacitor or inductor, apply its "
    "steady-state / t=0 behavior (a capacitor is open at steady state and a wire at t=0; an "
    "inductor is the reverse). Component values may be numbers (e.g. 100Ω, 12V) or symbols (e.g. "
    "R1, R2, V) — if they are symbols, give the answer as an algebraic expression in those symbols. "
    "Solve step by step, showing intermediate values (e.g. R_eq and each branch current), and "
    "end with a self-check. If a component value is not legible, say so and report the answer as "
    "null instead of guessing. End with a single line "
    "'FINAL: {\"quantity\":..., \"target_id\":..., \"value\":..., \"unit\":..., \"abstain\":false}'.")
CFG

{'MODEL': 'unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit',
 'MAX_IMAGE_PX': 512,
 'LORA_R': 16,
 'LORA_ALPHA': 16,
 'BATCH': 2,
 'GRAD_ACCUM': 4,
 'MAX_STEPS': 350,
 'EPOCHS': 1,
 'LR': 0.0002,
 'MAX_LEN': 2048,
 'SEED': 3407,
 'DATA_DIR': 'circuitsight_dataset',
 'SAVE_TO_DRIVE': False,
 'DRIVE_DIR': '/content/drive/MyDrive/CircuitSight',
 'OUT_DIR': 'circuitsight_qlora'}

## 3. Load the dataset

Get the dataset next to this notebook. Either mount Drive and point `DATA_DIR` at the unzipped folder, or upload the zip produced by the data-gen notebook.

In [5]:
import os, json, zipfile

# Path to the FRESHLY regenerated dataset zip (numeric + symbolic + mixed, current schema).
# Regenerate with the dataset-generation notebook, download circuitsight_dataset.zip, upload it here.
# Do NOT reuse circuitsight_dataset_11k_4_3.zip — it predates the current schema/families (no symbolic
# values, no gold_answer.symbolic flag) and the eval will mismatch.
ZIP_PATH = "/content/data_20260709_1_10k.zip"

# Unzip it. The dataset was zipped from INSIDE the dataset folder, so its contents
# (images/, train.jsonl, ...) sit at the zip root -> extract into CFG["DATA_DIR"].
assert os.path.exists(ZIP_PATH), f"{ZIP_PATH} not found - upload the fresh zip there first (folder icon, left sidebar)."
os.makedirs(CFG["DATA_DIR"], exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(CFG["DATA_DIR"])
print("unzipped into", CFG["DATA_DIR"])

IMG_DIR = os.path.join(CFG["DATA_DIR"], "images")
assert os.path.isdir(IMG_DIR), f"expected {IMG_DIR} after unzip - check the zip's internal structure"

def load_jsonl(name):
    p = os.path.join(CFG["DATA_DIR"], name)
    return [json.loads(l) for l in open(p)] if os.path.exists(p) else []

train_rows = load_jsonl("train.jsonl")
val_rows   = load_jsonl("val_synthetic.jsonl")
print(f"train: {len(train_rows)}  val: {len(val_rows)}")
print("example keys:", list(train_rows[0].keys()))
# sanity: confirm this is the FRESH dataset (has value_mode incl. symbolic/mixed)
from collections import Counter
print("value_mode mix:", Counter(r.get("value_mode","MISSING->stale zip!") for r in train_rows))

# --- alternatives to upload the zip ---
# from google.colab import files; up = files.upload(); ZIP_PATH = "/content/"+next(iter(up))
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/drive/MyDrive/CircuitSight/circuitsight_dataset.zip /content/

unzipped into circuitsight_dataset
train: 9500  val: 500
example keys: ['image', 'question', 'family', 'regime', 'question_type', 'value_mode', 'gold_components', 'gold_netlist', 'gold_topology', 'gold_answer', 'gold_values', 'render_style', 'skeleton', 'skills', 'concepts', 'gold_boxes', 'abstain', 'target_output']
value_mode mix: Counter({'numeric': 5271, 'symbolic': 2750, 'mixed': 1479})


## 4. Format as vision chat samples

Each row → a user turn (image + instruction + question) and an assistant turn (the worked solution). Images are loaded as RGB PIL and downsized to `MAX_IMAGE_PX` — passing real PIL images (not paths) avoids the common *'could not make a flat list of images'* collator error.

In [7]:
# All imports the training + eval cells rely on (safe to re-run anytime).
import os, json, zipfile, re, random, time
import torch
import numpy as np
from PIL import Image, ImageDraw
from datasets import Dataset
from collections import Counter

# unsloth / trl (already installed; just binding the names in this session)
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

print("all imports loaded — PIL, torch, datasets, unsloth, trl ready")

all imports loaded — PIL, torch, datasets, unsloth, trl ready


In [8]:
# Lazy image loading: keep only lightweight text rows in RAM; decode each image
# on demand when the trainer pulls it. This avoids holding 11k decoded images at once.
def load_image(name):
    img = Image.open(os.path.join(IMG_DIR, name)).convert("RGB")
    m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m / max(img.size)
        img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    return img

def to_conversation(row):
    return {"messages": [
        {"role":"user","content":[
            {"type":"image","image": load_image(row["image"])},
            {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + row["question"]}]},
        {"role":"assistant","content":[{"type":"text","text": row["target_output"]}]},
    ]}

class LazyConvDataset:
    """Builds each conversation (and loads its image) only when accessed."""
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        if isinstance(i, slice):
            return LazyConvDataset(self.rows[i])   # slicing -> a smaller lazy dataset
        return to_conversation(self.rows[i])
    def select(self, idxs):                        # HF-style helper, used by some trainers
        return LazyConvDataset([self.rows[k] for k in idxs])

train_conv = LazyConvDataset(train_rows)
print("lazy dataset ready:", len(train_conv), "samples (images load on demand)")
print("sample user text:\n", train_conv[0]["messages"][0]["content"][1]["text"][:200])

# def load_image(name):
#     img = Image.open(os.path.join(IMG_DIR, name)).convert("RGB")
#     m = CFG["MAX_IMAGE_PX"]
#     if max(img.size) > m:
#         s = m / max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
#     return img

# def to_conversation(row):
#     return {"messages": [
#         {"role":"user","content":[
#             {"type":"image","image": load_image(row["image"])},
#             {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + row["question"]}]},
#         {"role":"assistant","content":[{"type":"text","text": row["target_output"]}]},
#     ]}

# train_conv = [to_conversation(r) for r in train_rows]
# print("formatted", len(train_conv), "samples")
# print("sample user text:\n", train_conv[0]["messages"][0]["content"][1]["text"][:200])

lazy dataset ready: 9500 samples (images load on demand)
sample user text:
 You are a circuit analysis tutor. Look at the schematic and answer the question. First list every component, each with the image region it occupies as <box>[x0,y0,x1,y1]</box> in 0-1000 normalized coo


## 5. Load model + attach LoRA

We finetune both vision and language layers: component *identification* is a vision-side skill, equation *setup* is language-side, and this task needs both.

In [9]:
model, tokenizer = FastVisionModel.from_pretrained(
    CFG["MODEL"], load_in_4bit=True, use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=CFG["LORA_R"], lora_alpha=CFG["LORA_ALPHA"], lora_dropout=0,
    bias="none", random_state=CFG["SEED"])
print("model + LoRA ready")

Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

model + LoRA ready


## 6. Day-1 OOM smoke test — the go/no-go gate

**Run this before the full training.** It trains 5 steps on a handful of samples and reports peak VRAM. Purpose: find out *today* whether the model trains without OOM at your image resolution, instead of discovering it 3 days in.

- **Passes** (completes, peak VRAM leaves headroom) → proceed to full training.
- **OOMs** → in order: (1) drop `MAX_IMAGE_PX` to 512 and re-run cell 4; (2) set `finetune_vision_layers=False`; (3) switch `CFG["MODEL"]` to a 2B (e.g. SmolVLM) and reload cell 5. Re-run this gate until it passes.


In [ ]:
import time
FastVisionModel.for_training(model)
smoke = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=LazyConvDataset(train_rows[:8]),
    args=SFTConfig(
        per_device_train_batch_size=CFG["BATCH"], gradient_accumulation_steps=1,
        warmup_steps=0, max_steps=5, learning_rate=CFG["LR"], logging_steps=1,
        optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
        seed=CFG["SEED"], output_dir="smoke_out", report_to="none",
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True}, max_length=CFG["MAX_LEN"],
    ))
torch.cuda.reset_peak_memory_stats()
t=time.time(); smoke.train(); dt=time.time()-t
peak = torch.cuda.max_memory_reserved()/1e9
total = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"\nSMOKE TEST PASSED: 5 steps in {dt:.0f}s | peak VRAM {peak:.1f} / {total:.1f} GB")
print("GO." if peak < 0.9*total else "TIGHT - lower MAX_IMAGE_PX before the full run.")

Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8 | Num Epochs = 1 | Total steps = 5
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 41,084,928 of 3,795,707,904 (1.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.722866
2,2.113713
3,2.779430
4,2.187795
5,1.609380


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in smoke_out/checkpoint-5/tokenizer_config.json.



SMOKE TEST PASSED: 5 steps in 91s | peak VRAM 5.7 / 15.6 GB
GO.


## 7. Full training

**This is capped by `CFG["MAX_STEPS"]`, not epochs.** A full epoch over ~10k images at effective batch 4 is ~2,600 optimizer steps (≈6 h on a T4) — that's *steps*, not epochs. This behavior is highly structured and narrow, so a few hundred steps is plenty. `MAX_STEPS=150` at `MAX_IMAGE_PX=512` runs in roughly **15–25 min on a T4**. The trainer shuffles the full dataset, so 150 steps still samples across all three families.

**Scale strategy:** run 150 steps, look at the section-9 eval curve, and only raise `MAX_STEPS` (or set it to `None` for a full epoch) if accuracy is still climbing — extra steps past diminishing returns just burn time. If you OOM or it's too slow, keep `MAX_IMAGE_PX=512`; if you have headroom and want sharper boxes, try 640/768.

In [10]:
FastVisionModel.for_training(model)
eff_batch = CFG["BATCH"]*CFG["GRAD_ACCUM"]
use_steps = CFG.get("MAX_STEPS")
schedule  = dict(max_steps=use_steps) if use_steps else dict(num_train_epochs=CFG["EPOCHS"])
seen      = use_steps*eff_batch if use_steps else len(train_conv)
print(f"training: {'max_steps='+str(use_steps) if use_steps else 'epochs='+str(CFG['EPOCHS'])}"
      f" | effective batch {eff_batch} | ~{seen} images seen (dataset has {len(train_conv)}, shuffled)")

# periodic checkpoints so a disconnect does not lose the run (keeps the 2 most recent)
if CFG.get("SAVE_TO_DRIVE"):
    from google.colab import drive; drive.mount("/content/drive")
    CKPT_DIR = os.path.join(CFG["DRIVE_DIR"], "checkpoints")   # written straight to Drive during training
else:
    CKPT_DIR = CFG["OUT_DIR"]                                    # /content/circuitsight_qlora (download manually)
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"checkpoints -> {CKPT_DIR}  (every 100 steps, keeping the 2 most recent)")

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_conv,
    args=SFTConfig(
        per_device_train_batch_size=CFG["BATCH"], gradient_accumulation_steps=CFG["GRAD_ACCUM"],
        warmup_steps=5, learning_rate=CFG["LR"],
        logging_steps=10, optim="adamw_8bit", weight_decay=0.001, lr_scheduler_type="linear",
        seed=CFG["SEED"], output_dir=CKPT_DIR, report_to="none",
        remove_unused_columns=False, dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True}, max_length=CFG["MAX_LEN"],
        save_strategy="steps", save_steps=100, save_total_limit=2,   # mid-run checkpoints for disconnect safety (keep 2 newest)
        **schedule,
    ))
stats = trainer.train()
print("done. final loss:", round(stats.training_loss, 4))

training: max_steps=350 | effective batch 8 | ~2800 images seen (dataset has 9500, shuffled)
checkpoints -> circuitsight_qlora  (every 100 steps, keeping the 2 most recent)
Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,500 | Num Epochs = 1 | Total steps = 350
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,084,928 of 3,795,707,904 (1.08% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.907253
20,0.883608
30,0.339104
40,0.227024
50,0.150710
60,0.133147
70,0.135637
80,0.112446
90,0.102762
100,0.084213


Unsloth: Restored added_tokens_decoder metadata in circuitsight_qlora/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in circuitsight_qlora/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in circuitsight_qlora/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in circuitsight_qlora/checkpoint-350/tokenizer_config.json.


done. final loss: 0.1607


## 8. Inference helper

In [11]:
def solve_image(pil_img, question, model, tokenizer, max_new_tokens=400):
    FastVisionModel.for_inference(model)
    msgs = [{"role":"user","content":[{"type":"image"},
             {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]}]
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = tokenizer(pil_img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True,
                         do_sample=False, temperature=0.0)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()

# quick look
r0 = val_rows[0] if val_rows else train_rows[0]
print("Q:", r0["question"])
print(solve_image(load_image(r0["image"]), r0["question"], model, tokenizer)[:500])

Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Find the total current drawn from the source.
Components (with image regions): V1 <box>[734, 420, 804, 562]</box> (9V source); R1 <box>[496, 264, 562, 491]</box> (150Ω); R2 <box>[234, 264, 300, 491]</box> (1000Ω); R3 <box>[496, 491, 562, 718]</box> (10Ω); R4 <box>[234, 491, 300, 718]</box> (2200Ω); R5 <box>[267, 448, 529, 533]</box> (150Ω).
Concepts used: Ohm's law, Kirchhoff's current law (nodal analysis).
Topology: Wheatstone bridge — left branch R1 (top) & R3 (bottom), right branch R2 (top) & R4 (bottom), with R5 bridging the two midpoin


## 9. Evaluation harness (base vs. tuned)

This is the point of the whole project. The harness parses each model output (preferring the machine-readable `FINAL: {quantity, target_id, value, unit, abstain}` line, with a prose fallback) and scores it against the exact gold labels along separate axes:

- **component_accuracy** — right resistor count *and* the right component **types** present (R vs C vs L).
- **reactive_type_accuracy** — on capacitor/inductor circuits, did it identify the reactive component as the correct type (not confuse a capacitor for an inductor)? This is the headline **"spatial blindness"** number.
- **Req_accuracy / answer_accuracy / step_verified_accuracy** — did it get the final quantity right, *and* the `R_eq` intermediate right (a correct answer from a wrong intermediate does **not** count).
- **grounding_accuracy** (IoU ≥ 0.5 vs. gold boxes) and **grounding_hallucination_rate** — did the cited regions actually land on the components, or invent locations?
- **fabrication_rate** vs. **honest_abstention_rate** — on illegible-value cases, did it invent a number (bad) or correctly say null (good)?
- **concept_hallucination_rate** — did it invoke a solving concept the problem doesn't need?
- **answer_accuracy_by_qtype** — a breakdown across current / voltage / power / resistance / charge / energy.

We run it on the **base** model and the **tuned** model over the same held-out set and compare. Report the **real-world** transfer set (§10) as the headline number.


In [12]:
import re, json, random
import sympy as smp
# ============================================================================
# Eval harness for the generalized schema:
#   FINAL: {quantity, target_id, value, unit, abstain}
# Scores, separately, the axes the BrainLift cares about:
#   component identification (count + TYPE: R vs C vs L), grounding (IoU>=0.5) +
#   grounding-hallucination, step-verified solve (final AND R_eq intermediate),
#   fabrication vs honest-abstention, concept-hallucination, and — the headline
#   "spatial blindness" number — reactive component-type accuracy (cap vs inductor).
# ============================================================================
def parse_final(text):
    m = re.search(r"FINAL:\s*(\{.*\})", text, re.DOTALL)
    if m:
        try:
            j = json.loads(m.group(1))
            return {"quantity":j.get("quantity"),"target_id":j.get("target_id"),
                    "value":j.get("value"),"unit":j.get("unit"),"abstain":bool(j.get("abstain",False))}
        except Exception: pass
    low = text.lower()                                     # prose fallback
    ab = ("null" in low) or ("not legible" in low) or ("cannot" in low)
    val=None
    m2 = re.search(r"answer[^=]*=\s*[^=]*?(-?\d+(?:\.\d+)?(?:e-?\d+)?)", low)
    if m2:
        try: val=float(m2.group(1))
        except Exception: val=None
    return {"quantity":None,"target_id":None,"value":val,"unit":None,"abstain":ab}

def parse_components(text):
    res_ids = set(re.findall(r"\bR(\d+)\b", text))
    has_cap = bool(re.search(r"capacitor|\u00b5F|uF|\bC1\b", text, re.I))
    has_ind = bool(re.search(r"inductor|\bmH\b|\bL1\b", text, re.I))
    m = re.search(r"(\d+)\s*resistor", text.lower())
    return {"n_res": int(m.group(1)) if m else len(res_ids), "has_cap":has_cap, "has_ind":has_ind}

def parse_boxes(text):
    pat=r"(V\d+|R\w+|C\w+|L\w+|SW|VM)\s*<box>\s*\[?\s*(-?\d+)\s*,\s*(-?\d+)\s*,\s*(-?\d+)\s*,\s*(-?\d+)\s*\]?\s*</box>"
    return {cid:[int(a),int(b),int(c),int(d)] for cid,a,b,c,d in re.findall(pat, text)}

def _iou(p,g):
    ix0,iy0,ix1,iy1=max(p[0],g[0]),max(p[1],g[1]),min(p[2],g[2]),min(p[3],g[3])
    inter=max(0,ix1-ix0)*max(0,iy1-iy0)
    u=max(0,p[2]-p[0])*max(0,p[3]-p[1])+max(0,g[2]-g[0])*max(0,g[3]-g[1])-inter
    return inter/u if u>0 else 0.0

_CKEYS={"ohm":"ohm","parallel":"parallel","series":"series","capacitor":"capacitor","inductor":"inductor"}
def _concepts_text(text):
    m=re.search(r"concepts used:\s*(.+)", text.lower())
    if not m: return None
    chunk=m.group(1).split("\n")[0]
    return {v for k,v in _CKEYS.items() if k in chunk}
def _concepts_gold(concepts):
    s=" ".join(concepts).lower(); return {v for k,v in _CKEYS.items() if k in s}

_NUM=re.compile(r"-?\d+\.?\d*(?:[eE][+-]?\d+)?$")
_SYM_RNG=random.Random(12345)   # fixed seed -> reproducible symbolic-equivalence checks
def _is_num(x):
    return isinstance(x,(int,float)) or (isinstance(x,str) and bool(_NUM.match(x.strip())))
def sym_equal(a, b, trials=6):
    """Algebraic equivalence via random substitution: plug the same random reals into the
    shared free symbols and compare (robust to any equivalent form, e.g. R1+R2 == R2+R1)."""
    try:
        ea=smp.sympify(str(a)); eb=smp.sympify(str(b))
    except Exception:
        return str(a).replace(" ","")==str(b).replace(" ","")
    syms=sorted(ea.free_symbols | eb.free_symbols, key=str)
    if not syms:
        try: return abs(float(ea)-float(eb))<=1e-6*max(1.0,abs(float(eb)))
        except Exception:
            try: return smp.simplify(ea-eb)==0
            except Exception: return False
    for _ in range(trials):
        subsd={s:_SYM_RNG.uniform(1.5,9.5) for s in syms}
        try: fa=float(ea.subs(subsd)); fb=float(eb.subs(subsd))
        except Exception:
            try: return smp.simplify(ea-eb)==0
            except Exception: return False
        if abs(fa-fb) > 1e-6*max(1.0,abs(fb)): return False
    return True
def _close(pred, gold, rel=0.03):
    """Numeric (3% tol) when both numeric; else algebraic equivalence (symbolic answers)."""
    if pred is None or gold is None: return False
    if _is_num(pred) and _is_num(gold):
        return abs(float(pred)-float(gold)) <= max(1e-9, abs(float(gold))*rel)
    return sym_equal(pred, gold)

def score_record(gold, model_text, rel_tol=0.03, ground_mode="id"):
    ga=bool(gold["abstain"]); fam=gold.get("family","dc_resistor")
    p=parse_final(model_text); comp=parse_components(model_text); mb=parse_boxes(model_text)
    gc=gold["gold_components"]
    res={"family":fam,"question_type":gold.get("question_type"),
         "abstain_gold":ga,"abstain_pred":bool(p["abstain"]),
         "comp_ok":None,"react_type_ok":None,"req_ok":None,"answer_ok":None,"intermediates_ok":None,
         "fabricated":None,"honest_abstain":None,
         "concept_declared":None,"concept_hallucinated":None,
         "grounding_ok":None,"grounding_hallucinated":None}
    # component identification: right resistor COUNT and right presence of cap/inductor TYPE
    res["comp_ok"]=(comp["n_res"]==gc.get("resistor")) and \
                   (comp["has_cap"]==("capacitor" in gc)) and (comp["has_ind"]==("inductor" in gc))
    if fam=="reactive":                                    # cap-vs-inductor confusion (headline)
        want_cap="capacitor" in gc
        res["react_type_ok"]=bool((comp["has_cap"] and not comp["has_ind"]) if want_cap
                                  else (comp["has_ind"] and not comp["has_cap"]))
    # grounding: fraction of gold components boxed with IoU>=0.5; hallucination = boxes that miss
    gb=gold.get("gold_boxes",{})   # ground_mode="iou": id-agnostic greedy match (real images number their own way)
    if gb:
        if ground_mode=="iou":
            preds=list(mb.values()); used=[False]*len(preds); hit=0
            for g in gb.values():
                best=0.0; bi=-1
                for i,pbx in enumerate(preds):
                    if used[i]: continue
                    v=_iou(pbx,g)
                    if v>best: best=v; bi=i
                if bi>=0 and best>=0.5: used[bi]=True; hit+=1
            res["grounding_ok"]=hit/len(gb)
            if preds: res["grounding_hallucinated"]=sum(1 for u in used if not u)/len(preds)
        else:
            res["grounding_ok"]=sum(1 for cid,g in gb.items() if cid in mb and _iou(mb[cid],g)>=0.5)/len(gb)
            if mb:
                res["grounding_hallucinated"]=sum(1 for cid,m in mb.items()
                                                  if cid not in gb or _iou(m,gb[cid])<0.5)/len(mb)
    # concept scoping
    gset=_concepts_gold(gold.get("concepts",[])); pc=_concepts_text(model_text)
    if pc is None: res["concept_declared"]=False
    else: res["concept_declared"]=True; res["concept_hallucinated"]=len(pc-gset)>0
    # answer / abstention
    if ga:
        gave=(p["value"] is not None) and (not p["abstain"])
        res["fabricated"]=gave; res["honest_abstain"]=bool(p["abstain"]) and not gave
    else:
        ga_dict=gold.get("gold_answer") or {}          # perception-only records have gold_answer=None
        gv=ga_dict.get("value")                        # float | expr-string | None
        gvals=gold.get("gold_values") or {}
        res["answer_ok"]=_close(p["value"], gv, rel_tol) if gv is not None else None
        qt=gold.get("question_type")
        if qt=="resistance":
            res["req_ok"]=res["answer_ok"]
        elif qt in ("topology","components") or gv is None:   # perception-only: no numeric intermediate
            res["req_ok"]=None
        elif gvals.get("I_total")==0:                  # open circuit: no loop-R_eq intermediate
            res["req_ok"]=None
        elif "R_eq" in gvals:
            gold_req=gvals["R_eq"]
            if _is_num(gold_req):
                mR=re.search(r"r_eq\s*=\s*(-?\d+\.?\d*(?:[eE][+-]?\d+)?)", model_text, re.I)
                predR=mR.group(1) if mR else None
            else:
                mR=re.search(r"r_eq\s*=\s*([^\n]+?)\s*(?:ohm|\u03a9|\(=|$)", model_text, re.I) or \
                   re.search(r"r_eq\s*=\s*([^\n.]+)", model_text, re.I)
                predR=mR.group(1).strip().rstrip(".") if mR else None
            res["req_ok"]=_close(predR, gold_req, rel_tol)
        gbc=gvals.get("branch_currents",{})            # full intermediate verification
        if gbc:
            pbc={cid:v.strip() for cid,v in re.findall(r"I\((R\w+)\)\s*=\s*([^,\n]+?)\s*A", model_text)}
            checked=ok=0
            for cid,gvv in gbc.items():
                if _is_num(gvv) and abs(float(gvv))<1e-3: continue   # skip sub-mA (numeric rounding noise)
                checked+=1; pv=pbc.get(cid)
                if pv is not None and _close(pv, gvv, rel_tol): ok+=1
            res["intermediates_ok"]=(ok==checked) if checked else None
    return res

def aggregate(results):
    non=[r for r in results if not r["abstain_gold"]]; ab=[r for r in results if r["abstain_gold"]]
    rj =[r for r in results if r["family"]=="reactive"]
    def frac(xs,k):
        xs=[r[k] for r in xs if r[k] is not None]; return round(sum(xs)/len(xs),4) if xs else None
    step=[r for r in non if r["comp_ok"] and (r["req_ok"] in (True,None)) and (r["intermediates_ok"] in (True,None)) and r["answer_ok"]]
    qts=sorted(set(r["question_type"] for r in non if r["question_type"]))
    _gr=frac(results,"grounding_ok"); _gh=frac(results,"grounding_hallucinated")
    _gp=(round(1-_gh,4) if _gh is not None else None)
    _gf1=(round(2*_gp*_gr/(_gp+_gr),4) if (_gp and _gr and _gp+_gr>0) else None)
    _tp=sum(1 for r in results if r["abstain_gold"] and r["abstain_pred"])
    _fp=sum(1 for r in results if (not r["abstain_gold"]) and r["abstain_pred"])
    _fn=sum(1 for r in results if r["abstain_gold"] and not r["abstain_pred"])
    _apr=(round(_tp/(_tp+_fp),4) if _tp+_fp else None); _arc=(round(_tp/(_tp+_fn),4) if _tp+_fn else None)
    return {"n_total":len(results),"n_abstain":len(ab),"n_reactive":len(rj),
            "component_accuracy":frac(results,"comp_ok"),
            "reactive_type_accuracy":frac(rj,"react_type_ok"),
            "Req_accuracy":frac(non,"req_ok"),"answer_accuracy":frac(non,"answer_ok"),
            "intermediates_accuracy":frac(non,"intermediates_ok"),
            "step_verified_accuracy":round(len(step)/len(non),4) if non else None,
            "fabrication_rate":frac(ab,"fabricated"),"honest_abstention_rate":frac(ab,"honest_abstain"),
            "grounding_accuracy":frac(results,"grounding_ok"),
            "grounding_hallucination_rate":frac(results,"grounding_hallucinated"),
            "concept_declared_rate":frac(results,"concept_declared"),
            "concept_hallucination_rate":frac(results,"concept_hallucinated"),
            "grounding_recall":_gr,"grounding_precision":_gp,"grounding_f1":_gf1,
            "abstention_precision":_apr,"abstention_recall":_arc,
            "answer_accuracy_by_qtype":{q:frac([r for r in non if r["question_type"]==q],"answer_ok") for q in qts}}

def evaluate(model, tokenizer, rows, n=None, ground_mode="id"):
    # ground_mode="iou" for the real-world set (model numbers components its own way)
    rows = rows[:n] if n else rows
    return aggregate([score_record(r, solve_image(load_image(r["image"]), r["question"], model, tokenizer),
                                    ground_mode=ground_mode)
                      for r in rows])
print("eval harness loaded (generalized schema + grounding + component-type confusion)")

eval harness loaded (generalized schema + grounding + component-type confusion)


### §9f — Fine-tuned 3B vs a frontier VLM (same synthetic eval)

Head-to-head on the **held-out synthetic** val set — the fair arena for the moat. Both models receive the **identical image + `INSTRUCTION` + scoring harness**; the frontier model runs via the same TrueFoundry gateway as §9e (vision-capable Claude).

**Why synthetic, not AP FRQs:** a frontier model can web-search a *published* FRQ solution that matches the figure (de-facto tool offloading) — our offline 3B can't — so an exam question would measure retrieval, not reasoning. Procedurally-generated circuits have **no web solution**, isolating on-image **perception + grounding + reasoning** on the complex diagrams (up to 4 branches / 3 resistors per branch) where generic VLMs are documented to fail.

Run **§9** (harness + model load) first; running **§9e** first is ideal (reuses its gateway `client` so you don't re-enter the token). Grounding is scored **id-agnostically** (`ground_mode="iou"`) so neither model is penalized for how it numbers components.

In [ ]:
# ============================================================================
# §9f — Fine-tuned 3B  vs  a FRONTIER VLM, on the SAME held-out SYNTHETIC eval
# ----------------------------------------------------------------------------
# The fair arena for the moat: identical image + INSTRUCTION + scoring harness for
# BOTH models. We use SYNTHETIC (not AP FRQs) on purpose — a frontier model can
# web-search a *published* FRQ solution that matches the figure (de-facto tool
# offload) which our offline 3B cannot; procedurally-generated circuits have no web
# solution, so this isolates on-image perception + grounding + reasoning on the
# complex diagrams where generic VLMs are documented to fail.
# Frontier model = vision-capable Claude via the same TrueFoundry gateway as §9e.
# ============================================================================
try:
    import anthropic
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=False)
    import anthropic
import os, io, re, json, base64
from PIL import Image

FRONTIER_MODEL = "claude-sonnet-5"   # a vision model your gateway serves (§9e used this). Bump to a
                                     # stronger id (e.g. claude-opus-4-8) if your TFY dashboard serves it.
FRONTIER_N     = 20                  # frontier API calls (costs gateway credits). Raise for firmer numbers.
GROUND_MODE    = "iou"               # id-agnostic box overlap: fair, since the frontier numbers parts its own way.

# IMG_DIR / val_rows are set by §3 (Load the dataset). Reconstruct IMG_DIR if this session
# didn't set the variable but the unzipped files are on disk.
if "IMG_DIR" not in globals() and "CFG" in globals():
    IMG_DIR = os.path.join(CFG["DATA_DIR"], "images")
for _n in ["score_record","aggregate","solve_image","load_image","INSTRUCTION"]:
    assert _n in globals(), f"'{_n}' missing — run the §9 eval harness first."
assert "IMG_DIR" in globals() and os.path.isdir(IMG_DIR), \
    "image folder not found — run §3 (Load the dataset) first (it sets IMG_DIR + val_rows)."
assert ("val_rows" in globals()) or ("train_rows" in globals()), \
    "no val_rows/train_rows — run §3 (Load the dataset) first."
assert "model" in globals() and "tokenizer" in globals(), "Load the fine-tuned model (§9) first."

# --- gateway client: reuse §9e's if present, else build it (Anthropic SDK, Bearer auth) ---
if "client" not in globals():
    TFY_BASE_URL = "https://YOUR-TRUEFOUNDRY-GATEWAY"   # <- your gateway (Claude Code ANTHROPIC_BASE_URL)
    assert "YOUR-TRUEFOUNDRY" not in TFY_BASE_URL, "Set TFY_BASE_URL to your TrueFoundry gateway URL."
    def _get_token():
        for name in ("TFY_TOKEN","ANTHROPIC_AUTH_TOKEN","TRUEFOUNDRY_API_KEY"):
            try:
                from google.colab import userdata
                k = userdata.get(name)
                if k: return k
            except Exception: pass
        import getpass
        return os.environ.get("TFY_TOKEN") or getpass.getpass("TrueFoundry token (Bearer; hidden): ")
    client = anthropic.Anthropic(base_url=TFY_BASE_URL, auth_token=_get_token().strip())

# preflight so we fail fast on a bad model id / token (never prints the token)
try:
    client.messages.create(model=FRONTIER_MODEL, max_tokens=5, messages=[{"role":"user","content":"ping"}])
    print("gateway OK — frontier model:", FRONTIER_MODEL)
except Exception as e:
    raise RuntimeError(f"Frontier preflight FAILED for '{FRONTIER_MODEL}': {str(e)[:220]}\n"
        "-> set FRONTIER_MODEL to a vision model your TFY dashboard serves (e.g. claude-sonnet-5).")

def _img_b64(name, maxpx=768):
    im = Image.open(os.path.join(IMG_DIR, name)).convert("RGB")
    if max(im.size) > maxpx:
        s = maxpx/max(im.size); im = im.resize((int(im.size[0]*s), int(im.size[1]*s)))
    buf = io.BytesIO(); im.save(buf, format="JPEG", quality=90)
    return base64.standard_b64encode(buf.getvalue()).decode()

def solve_image_frontier(image_name, question):
    """Same task/prompt as solve_image, but solved by the frontier VLM (identical INSTRUCTION+image)."""
    content = [{"type":"image","source":{"type":"base64","media_type":"image/jpeg","data":_img_b64(image_name)}},
               {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]
    attempts = [dict(max_tokens=2000, temperature=0, thinking={"type":"disabled"}),   # clean, deterministic
                dict(max_tokens=5000, thinking={"type":"enabled","budget_tokens":2000}),
                dict(max_tokens=9000)]                                                 # gateway ignored 'thinking'
    last = ""
    for kw in attempts:
        try:
            msg = client.messages.create(model=FRONTIER_MODEL, messages=[{"role":"user","content":content}], **kw)
            txt = "".join(getattr(b,"text","") for b in msg.content if getattr(b,"type",None)=="text")
            if txt.strip(): return txt
            last = "empty text (thinking used the whole budget)"
        except Exception as e:
            last = str(e)[:160]
    return f"(frontier call failed: {last})"

# --- run BOTH on the same synthetic rows ---
rows = (val_rows or train_rows[-FRONTIER_N:])[:FRONTIER_N]
print(f"comparing on {len(rows)} synthetic val items (ground_mode={GROUND_MODE})\n")
ours_res, front_res = [], []
for i, r in enumerate(rows, 1):
    o_txt = solve_image(load_image(r["image"]), r["question"], model, tokenizer)
    f_txt = solve_image_frontier(r["image"], r["question"])
    ours_res.append(score_record(r, o_txt, ground_mode=GROUND_MODE))
    front_res.append(score_record(r, f_txt, ground_mode=GROUND_MODE))
    print(f"  [{i}/{len(rows)}] {r['image']:>16}  ({r.get('family')})")
ours_m, front_m = aggregate(ours_res), aggregate(front_res)

# --- side-by-side ---
KEYS = ["component_accuracy","reactive_type_accuracy","grounding_recall","grounding_precision",
        "grounding_hallucination_rate","Req_accuracy","intermediates_accuracy","answer_accuracy",
        "step_verified_accuracy","concept_hallucination_rate","honest_abstention_rate","fabrication_rate"]
print(f"\n=== Fine-tuned 3B  vs  {FRONTIER_MODEL}   (synthetic, n={len(rows)}) ===")
print(f"{'metric':32}{'ours(3B)':>10}{'frontier':>10}   winner")
for k in KEYS:
    o, f = ours_m.get(k), front_m.get(k)
    if o is None and f is None: continue
    lower_better = ("hallucination" in k) or ("fabrication" in k)
    os_ = "n/a" if o is None else f"{o:.3f}"; fs_ = "n/a" if f is None else f"{f:.3f}"
    win = ""
    if o is not None and f is not None:
        win = "tie" if o == f else ("ours" if ((o < f) if lower_better else (o > f)) else "frontier")
    print(f"{k:32}{os_:>10}{fs_:>10}   {win}")
print("\nanswer accuracy by value mode:")
print("  ours    :", ours_m.get("answer_accuracy_by_value_mode"))
print("  frontier:", front_m.get("answer_accuracy_by_value_mode"))
print(
    "\nRead: both get the identical image + INSTRUCTION + harness; grounding is id-agnostic "
    f"(ground_mode='{GROUND_MODE}'), so neither is penalized for how it labels parts. Expect the frontier "
    "model to lead on final-number arithmetic while the tuned 3B leads on grounding + structured, "
    "checkable output — the moat is reliable on-image perception, not being a bigger calculator.")


### §9g — Pick & visualize the demo example (base Qwen vs Sonnet vs tuned 3B)

Scans the hardest synthetic circuits and finds one where the **tuned 3B grounds + identifies correctly but both base Qwen and frontier Sonnet fail** — the moat in a single image — then overlays each model's predicted boxes on the image (saved to `demo_boxes.png`) and prints the three raw outputs for the talk track. Run **§9** (base vs tuned) and **§9f** (Sonnet) first. See [docs/demo_script.md](../docs/demo_script.md) for the narration.

In [ ]:
# ============================================================================
# §9g — Visualize the demo example: GOLD vs base Qwen vs frontier Sonnet vs tuned 3B
# ----------------------------------------------------------------------------
# Renders a 4-panel box overlay on ONE image — ground truth + each model's predicted
# boxes — and saves it to demo_boxes.png (absolute path printed). Pin FORCE_IMAGE to
# the moat example you found (skips the scan + saves Sonnet calls), or set it to None
# to re-scan the hardest circuits for one where the tuned model wins and both others fail.
# Deps: §9 (base_model+base_tok+model+tokenizer, harness+parse_boxes), §9f (solve_image_frontier).
# ============================================================================
import os, matplotlib.pyplot as plt
assert "base_model" in globals() and "base_tok" in globals(), "run the §9 base-vs-tuned cell first."
assert "solve_image_frontier" in globals(), "run §9f first (defines solve_image_frontier)."
assert "parse_boxes" in globals() and "score_record" in globals(), "run the §9 eval harness first."

FORCE_IMAGE = "img_002430.jpg"   # pin the moat image from the scan (None = re-scan). Skips Sonnet calls.
DEMO_SCAN   = 12                 # candidates to scan when FORCE_IMAGE is None

def _run3(r):
    return {"tuned":  solve_image(load_image(r["image"]), r["question"], model, tokenizer),
            "base":   solve_image(load_image(r["image"]), r["question"], base_model, base_tok),
            "sonnet": solve_image_frontier(r["image"], r["question"])}
def _score(r, texts):
    return {k: score_record(r, v, ground_mode="iou") for k, v in texts.items()}

best = None
if FORCE_IMAGE and any(x["image"] == FORCE_IMAGE for x in val_rows):
    r = next(x for x in val_rows if x["image"] == FORCE_IMAGE)
    t = _run3(r); best = dict(r=r, texts=t, scores=_score(r, t))
    print(f"using pinned image: {FORCE_IMAGE}  ({r['family']})")
else:
    if FORCE_IMAGE: print(f"{FORCE_IMAGE} not in val_rows — scanning instead")
    cands  = [r for r in val_rows if r.get("family") == "bridge"]
    cands += [r for r in val_rows if r.get("family") == "dc_resistor"
              and r.get("gold_components", {}).get("resistor", 0) >= 4]
    cands = cands[:DEMO_SCAN]
    print(f"scanning {len(cands)} complex candidates...\n")
    for r in cands:
        t = _run3(r); sc = _score(r, t)
        g = {k: (sc[k]["grounding_ok"] or 0.0) for k in sc}
        tuned_ok  = g["tuned"] >= 0.8 and sc["tuned"]["comp_ok"]
        others_no = (g["base"] < 0.5 or not sc["base"]["comp_ok"]) and (g["sonnet"] < 0.5 or not sc["sonnet"]["comp_ok"])
        gap = g["tuned"] - max(g["base"], g["sonnet"])
        tag = "  <== MOAT" if (tuned_ok and others_no) else ""
        print(f"  {r['image']:>16} {r['family']:11} ground b/s/t={g['base']:.2f}/{g['sonnet']:.2f}/{g['tuned']:.2f}{tag}")
        if tuned_ok and others_no and (best is None or gap > best["gap"]):
            best = dict(gap=gap, r=r, texts=t, scores=sc)
    assert best, "no clean 3-way moat example — raise DEMO_SCAN or set FORCE_IMAGE."

# ---- 4-panel overlay: GOLD (green) + base / sonnet / tuned (red) ----
r = best["r"]; img = load_image(r["image"]); W, H = img.size
print(f"\n=== {r['image']}  ({r['family']}) ===\nQ: {r['question']}\n")
panels  = [("Ground truth", r.get("gold_boxes", {}), "lime", None)]
panels += [(lbl, parse_boxes(best["texts"][k]), "red", best["scores"][k]["grounding_ok"])
           for k, lbl in [("base", "Base Qwen-3B"), ("sonnet", "Frontier Sonnet"), ("tuned", "Tuned 3B (ours)")]]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, bx, color, g) in zip(axes, panels):
    ax.imshow(img); ax.set_axis_off()
    for cid, (x0, y0, x1, y1) in bx.items():
        ax.add_patch(plt.Rectangle((x0/1000*W, y0/1000*H), (x1-x0)/1000*W, (y1-y0)/1000*H,
                                   fill=False, edgecolor=color, lw=2.2))
        ax.text(x0/1000*W, max(0, y0/1000*H - 3), cid, color=color, fontsize=8, weight="bold")
    sub = "" if g is None else f"\ngrounding={g:.2f}   boxes={len(bx)}"
    ax.set_title(title + sub, fontsize=12)
plt.tight_layout()
out = os.path.abspath("demo_boxes.png"); plt.savefig(out, dpi=130, bbox_inches="tight"); plt.show()
print("\nsaved ->", out)
# try: from google.colab import files; files.download(out)   # uncomment to auto-download the PNG

for k, lbl in [("base", "BASE QWEN-3B"), ("sonnet", "FRONTIER SONNET"), ("tuned", "TUNED 3B (OURS)")]:
    sc = best["scores"][k]
    print(f"\n----- {lbl}  (comp_ok={sc['comp_ok']}, grounding={sc['grounding_ok']}) -----")
    print(best["texts"][k][:650].strip())


In [19]:
import gc, torch
# shorter generations (our targets are ~150-350 tokens; 256 is enough) -> ~2x faster per image
def solve_image(pil_img, question, model, tokenizer, max_new_tokens=256):
    FastVisionModel.for_inference(model)
    msgs=[{"role":"user","content":[{"type":"image"},{"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]}]
    text=tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs=tokenizer(pil_img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out=model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False, temperature=0.0)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()

EVAL_ROWS = val_rows if val_rows else train_rows[-40:]
EVAL_N = 24                                   # 24 tuned + 24 base ≈ ~5-8 min
tuned = evaluate(model, tokenizer, EVAL_ROWS, EVAL_N); print("TUNED:", tuned)

if "base_model" in globals(): del base_model
gc.collect(); torch.cuda.empty_cache()
base_model, base_tok = FastVisionModel.from_pretrained(CFG["MODEL"], load_in_4bit=True, use_gradient_checkpointing="unsloth")
FastVisionModel.for_inference(base_model)
base = evaluate(base_model, base_tok, EVAL_ROWS, EVAL_N); print("BASE:", base)

print("\n=== DELTA (tuned - base) ===")
for k in tuned:
    if isinstance(tuned[k],(int,float)) and isinstance(base.get(k),(int,float)):
        print(f"{k:28s}: {base[k]:.3f} -> {tuned[k]:.3f}  ({tuned[k]-base[k]:+.3f})")


### below runs for too long ###
# # Base vs tuned on the held-out synthetic val set (use the real-world eval set for the headline number).
# EVAL_ROWS = val_rows if val_rows else train_rows[-40:]
# EVAL_N = min(60, len(EVAL_ROWS))

# # --- tuned (current, fine-tuned model) ---
# tuned_metrics = evaluate(model, tokenizer, EVAL_ROWS, EVAL_N)
# print("TUNED :", tuned_metrics)

# # --- base (reload a fresh, un-tuned model) ---
# base_model, base_tok = FastVisionModel.from_pretrained(
#     CFG["MODEL"], load_in_4bit=True, use_gradient_checkpointing="unsloth")
# FastVisionModel.for_inference(base_model)
# base_metrics = evaluate(base_model, base_tok, EVAL_ROWS, EVAL_N)
# print("BASE  :", base_metrics)

# print("\n=== DELTA (tuned - base) ===")
# for k in tuned_metrics:
#     if isinstance(tuned_metrics[k],(int,float)) and isinstance(base_metrics.get(k),(int,float)):
#         print(f"{k:28s}: {base_metrics[k]:.3f} -> {tuned_metrics[k]:.3f}  ({tuned_metrics[k]-base_metrics[k]:+.3f})")

# # per-question-type answer accuracy (where the tuned model helps most)
# print("\n--- answer accuracy by question type (base -> tuned) ---")
# bt = base_metrics.get("answer_accuracy_by_qtype",{}); tt = tuned_metrics.get("answer_accuracy_by_qtype",{})
# for q in sorted(set(bt)|set(tt)):
#     b,t = bt.get(q), tt.get(q)
#     bs = "  n/a" if b is None else f"{b:.3f}"; ts = "  n/a" if t is None else f"{t:.3f}"
#     print(f"  {q:12s}: {bs} -> {ts}")

Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

TUNED: {'n_total': 24, 'n_abstain': 1, 'n_reactive': 6, 'component_accuracy': 0.9583, 'reactive_type_accuracy': 1.0, 'Req_accuracy': 0.1818, 'answer_accuracy': 0.087, 'intermediates_accuracy': 0.1818, 'step_verified_accuracy': 0.087, 'fabrication_rate': 0.0, 'honest_abstention_rate': 1.0, 'grounding_accuracy': 0.8417, 'grounding_hallucination_rate': 0.1525, 'concept_declared_rate': 0.8333, 'concept_hallucination_rate': 0.05, 'grounding_recall': 0.8417, 'grounding_precision': 0.8475, 'grounding_f1': 0.8446, 'abstention_precision': 1.0, 'abstention_recall': 1.0, 'answer_accuracy_by_qtype': {'charge': 0.0, 'current': 0.125, 'energy': 0.0, 'power': 0.0, 'resistance': 0.0, 'voltage': 0.2}}


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

==((====))==  Unsloth 2026.7.2: Fast Qwen2_5_Vl patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

BASE: {'n_total': 24, 'n_abstain': 1, 'n_reactive': 6, 'component_accuracy': 0.1667, 'reactive_type_accuracy': 0.3333, 'Req_accuracy': 0.0, 'answer_accuracy': 0.0, 'intermediates_accuracy': 0.0, 'step_verified_accuracy': 0.0, 'fabrication_rate': 0.0, 'honest_abstention_rate': 0.0, 'grounding_accuracy': 0.0, 'grounding_hallucination_rate': None, 'concept_declared_rate': 0.0, 'concept_hallucination_rate': None, 'grounding_recall': 0.0, 'grounding_precision': None, 'grounding_f1': None, 'abstention_precision': None, 'abstention_recall': 0.0, 'answer_accuracy_by_qtype': {'charge': 0.0, 'current': 0.0, 'energy': 0.0, 'power': 0.0, 'resistance': 0.0, 'voltage': 0.0}}

=== DELTA (tuned - base) ===
n_total                     : 24.000 -> 24.000  (+0.000)
n_abstain                   : 1.000 -> 1.000  (+0.000)
n_reactive                  : 6.000 -> 6.000  (+0.000)
component_accuracy          : 0.167 -> 0.958  (+0.792)
reactive_type_accuracy      : 0.333 -> 1.000  (+0.667)
Req_accuracy           

In [6]:
from google.colab import drive; drive.mount('/content/drive')
import glob
# if you trained with SAVE_TO_DRIVE=True, the snapshot is already here — no upload needed:
print(glob.glob('/content/drive/MyDrive/slm_project/models/*.zip'))
MODEL_ZIP = '/content/drive/MyDrive/slm_project/models/circuitsight_qlora_final_20260709_1.zip'  # <- pick from the list


Mounted at /content/drive
['/content/drive/MyDrive/slm_project/models/circuitsight_qlora_final_20260709_1.zip']


In [13]:
# Re-run-safe value-mode split — NO monkeypatching (that's what caused the recursion).
# STEP 1 FIRST: re-run the §9 harness cell so score_record is the clean original again.
from collections import Counter
EVAL_ROWS = val_rows if val_rows else train_rows[-40:]
EVAL_N_SPLIT = min(36, len(EVAL_ROWS))
rows = EVAL_ROWS[:EVAL_N_SPLIT]
print("scoring TUNED on", len(rows), "rows | mix:",
      dict(Counter(r.get("value_mode","numeric") for r in rows if not r.get("abstain"))))

scored = []
for i, r in enumerate(rows, 1):
    txt = solve_image(load_image(r["image"]), r["question"], model, tokenizer)
    s = score_record(r, txt)                       # original scorer (restored in step 1)
    s["value_mode"] = r.get("value_mode", "numeric")
    scored.append(s)
    if i % 10 == 0: print(f"  ...{i}/{len(rows)}")

def _acc(xs):
    xs = [s["answer_ok"] for s in xs if s.get("answer_ok") is not None]
    return round(sum(xs)/len(xs), 4) if xs else None

non = [s for s in scored if not s["abstain_gold"]]
print(f"\noverall answer_accuracy: {_acc(non)}")
print("--- TUNED answer accuracy by value mode ---")
for vm in ("numeric","symbolic","mixed"):
    sub = [s for s in non if s.get("value_mode")==vm]
    print(f"  {vm:9s}: {'n/a' if _acc(sub) is None else f'{_acc(sub):.3f}'}   (n={len(sub)})")


Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


scoring TUNED on 36 rows | mix: {'numeric': 14, 'symbolic': 13, 'mixed': 6}


Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  ...10/36


Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  ...20/36


Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  ...30/36


Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=128000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


overall answer_accuracy: 0.1818
--- TUNED answer accuracy by value mode ---
  numeric  : 0.000   (n=14)
  symbolic : 0.308   (n=13)
  mixed    : 0.333   (n=6)


## 10. Real-world evaluation (the headline number)

Synthetic accuracy overstates real performance. Fill in the hand-labeled real set (`real_eval_TEMPLATE.json` from the data-gen notebook), load it the same way, and run `evaluate(...)` on it. Report **base vs. tuned on the real set** as your main result, and report the synthetic→real gap honestly.

In [13]:
# Real-world transfer eval (the headline number). Upload the hand-labeled set to REAL_DIR — the
# whole data/real_eval/ folder, keeping its subpaths (labeled.jsonl + figures/... + opensource/...).
REAL_DIR  = "/content/real_eval"
real_path = os.path.join(REAL_DIR, "labeled.jsonl")

def _real_img(name):                                  # images live under REAL_DIR (not the train IMG_DIR)
    img = Image.open(os.path.join(REAL_DIR, name)).convert("RGB")
    m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m/max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    return img

def eval_real(mdl, tok):
    # ground_mode="iou": match predicted->gold boxes by IoU regardless of id (the model numbers
    # real components its own way, so "R1" vs gold "Rx" must not fail grounding).
    res=[score_record(r, solve_image(_real_img(r["image"]), r["question"], mdl, tok), ground_mode="iou")
         for r in real_rows]
    return aggregate(res)

if os.path.exists(real_path):
    real_rows=[json.loads(l) for l in open(real_path)]
    for r in real_rows: r.setdefault("abstain", False)
    print(f"real records: {len(real_rows)}")
    print("REAL tuned:", eval_real(model, tokenizer))
    print("REAL base :", eval_real(base_model, base_tok))
    # NOTE: 6/8 real records are symbolic (perception-only) -> read component_accuracy,
    # reactive_type_accuracy, grounding_accuracy; answer/step-verified apply to the numeric ones.
else:
    print(f"No {real_path} yet - upload data/real_eval/ there (labeled.jsonl + figures/ + opensource/).")

No /content/real_eval/labeled.jsonl yet - upload data/real_eval/ there (labeled.jsonl + figures/ + opensource/).


## 11. Save / push the adapter

In [15]:
model.save_pretrained(CFG["OUT_DIR"]); tokenizer.save_pretrained(CFG["OUT_DIR"])
print("saved LoRA adapter to", CFG["OUT_DIR"])

if CFG.get("SAVE_TO_DRIVE"):
    import os, shutil, time, glob
    from google.colab import drive; drive.mount("/content/drive")
    models_dir = os.path.join(CFG["DRIVE_DIR"], "models"); os.makedirs(models_dir, exist_ok=True)
    stamp = time.strftime("%Y%m%d_%H%M%S")
    snap = shutil.make_archive(os.path.join(models_dir, f"circuitsight_qlora_{stamp}"), "zip", CFG["OUT_DIR"])
    print("model snapshot saved to Drive:", snap)
    # keep only the 2 most recent snapshots so Drive does not fill up
    snaps = sorted(glob.glob(os.path.join(models_dir, "circuitsight_qlora_*.zip")), key=os.path.getmtime)
    for old in snaps[:-2]:
        os.remove(old); print("  pruned old snapshot:", os.path.basename(old))
    print("  kept:", [os.path.basename(s) for s in snaps[-2:]])
# To the Hub:
# from huggingface_hub import login; login()
# model.push_to_hub("your-username/circuitsight-qwen2.5vl-3b")
# tokenizer.push_to_hub("your-username/circuitsight-qwen2.5vl-3b")
# Merged 16-bit for deployment:
# model.save_pretrained_merged("circuitsight_merged", tokenizer)

Unsloth: Restored added_tokens_decoder metadata in circuitsight_qlora/tokenizer_config.json.


saved LoRA adapter to circuitsight_qlora


In [16]:
# Save the in-memory fine-tuned model and zip it for local download (no retrain).
import os, shutil
OUT = "circuitsight_qlora"
model.save_pretrained(OUT); tokenizer.save_pretrained(OUT)
os.makedirs("/content/downloads", exist_ok=True)
z = shutil.make_archive("/content/downloads/circuitsight_qlora_final", "zip", OUT)
print("zip ready:", z, "->", round(os.path.getsize(z)/1e6, 1), "MB")
print("Download: folder icon (left sidebar) -> downloads -> right-click circuitsight_qlora_final.zip -> Download")


zip ready: /content/downloads/circuitsight_qlora_final.zip -> 610.6 MB
Download: folder icon (left sidebar) -> downloads -> right-click circuitsight_qlora_final.zip -> Download


## 12. For graders — load the fine-tuned model and run it on one image

No HuggingFace account needed. In a **fresh runtime**, the minimum path is:
1. Run **§1** (install) → **Restart session** → the **imports** cell → the **§2 Config** cell (defines `CFG` + `INSTRUCTION`).
2. Get the adapter: unzip the submitted `circuitsight_qlora_*.zip` and point `ADAPTER_DIR` at the unzipped folder (it contains `adapter_config.json`).
3. Set `IMAGE_PATH` to a circuit image (upload one via the file browser) and run the cell below.

Right after training (same session) it just reuses the model already in memory.

In [17]:
# ---- Load a saved model .zip and run the fine-tuned model ----
ZIP_PATH   = "/content/circuitsight_qlora_final.zip"   # <- upload your zip here (final adapter OR a checkpoint-*.zip)
IMAGE_PATH = ""                                         # <- a circuit image to test (upload via the file browser)
QUESTION   = "Identify the components (with their image regions), state the topology, and solve the circuit."

import os, glob, zipfile
from unsloth import FastVisionModel
from PIL import Image

# unzip, then locate the adapter (works whether it sits at the zip root or one level down)
UNZIP_DIR = "/content/loaded_model"
os.makedirs(UNZIP_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(UNZIP_DIR)
hits = glob.glob(os.path.join(UNZIP_DIR, "**", "adapter_config.json"), recursive=True)
assert hits, f"no adapter_config.json under {UNZIP_DIR} - wrong zip?"
ADAPTER_DIR = os.path.dirname(hits[0]); print("adapter at:", ADAPTER_DIR)

_m, _t = FastVisionModel.from_pretrained(ADAPTER_DIR, load_in_4bit=True, use_gradient_checkpointing="unsloth")
FastVisionModel.for_inference(_m)
print("loaded fine-tuned model")

INSTR = globals().get("INSTRUCTION") or (
    "You are a circuit analysis tutor. Look at the schematic and answer the question. First list every "
    "component with its image region as <box>[x0,y0,x1,y1]</box> in 0-1000 coords. State the concepts and "
    "topology. If there is a capacitor/inductor apply its steady-state / t=0 behavior. Values may be numbers "
    "or symbols (answer with an expression if symbolic). Solve step by step showing intermediates (R_eq, each "
    "branch current), then a self-check. End with a single line "
    "'FINAL: {\"quantity\":..., \"target_id\":..., \"value\":..., \"unit\":..., \"abstain\":false}'.")
MAXPX = CFG["MAX_IMAGE_PX"] if "CFG" in globals() else 512

def run_circuit(image_path, question=QUESTION, max_new_tokens=400):
    img = Image.open(image_path).convert("RGB")
    if max(img.size) > MAXPX:
        s = MAXPX/max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    msgs=[{"role":"user","content":[{"type":"image"},{"type":"text","text": INSTR + "\n\nQuestion: " + question}]}]
    text=_t.apply_chat_template(msgs, add_generation_prompt=True)
    inputs=_t(img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out=_m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0, use_cache=True)
    return _t.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()

# score on a labeled set (if the §9 harness + rows are loaded): evaluate(_m, _t, val_rows, ground_mode="id")
if IMAGE_PATH:
    print("\nQ:", QUESTION, "\n"); print(run_circuit(IMAGE_PATH))
else:
    print("\nSet IMAGE_PATH to a circuit image and re-run (or call run_circuit('/path.png')).")


FileNotFoundError: [Errno 2] No such file or directory: '/content/circuitsight_qlora_final.zip'

In [14]:
# ---- FOR GRADERS: reload the fine-tuned model and run it on ONE circuit image ----
ADAPTER_DIR = CFG["OUT_DIR"]   # folder with adapter_config.json; or an unzipped circuitsight_qlora_*.zip snapshot
IMAGE_PATH  = ""               # <- set to your circuit image, e.g. "/content/my_circuit.png"
QUESTION    = "Identify the components (with their image regions), state the topology, and solve the circuit."

from unsloth import FastVisionModel
from PIL import Image

if "model" in globals() and "tokenizer" in globals():
    _m, _t = model, tokenizer                      # reuse the model from this training session (no extra VRAM)
    print("using the in-session fine-tuned model")
else:                                              # fresh runtime: load base (4-bit) + our LoRA adapter
    _m, _t = FastVisionModel.from_pretrained(ADAPTER_DIR, load_in_4bit=True,
                                             use_gradient_checkpointing="unsloth")
    print("loaded fine-tuned model from", ADAPTER_DIR)
    # fallback if your Unsloth version won't load an adapter dir directly:
    #   from peft import PeftModel
    #   _m, _t = FastVisionModel.from_pretrained(CFG["MODEL"], load_in_4bit=True)
    #   _m = PeftModel.from_pretrained(_m, ADAPTER_DIR)
FastVisionModel.for_inference(_m)

def run_circuit(image_path, question=QUESTION, max_new_tokens=400):
    img = Image.open(image_path).convert("RGB")
    m = CFG["MAX_IMAGE_PX"]
    if max(img.size) > m:
        s = m/max(img.size); img = img.resize((int(img.size[0]*s), int(img.size[1]*s)))
    msgs=[{"role":"user","content":[{"type":"image"},
           {"type":"text","text": INSTRUCTION + "\n\nQuestion: " + question}]}]
    text=_t.apply_chat_template(msgs, add_generation_prompt=True)
    inputs=_t(img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out=_m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0, use_cache=True)
    return _t.decode(out[0], skip_special_tokens=True).split("assistant")[-1].strip()

if IMAGE_PATH:
    print("\nQ:", QUESTION, "\n")
    print(run_circuit(IMAGE_PATH))
else:
    print("\nSet IMAGE_PATH to a circuit image (upload one via the file browser) and re-run this cell.")

using the in-session fine-tuned model

Set IMAGE_PATH to a circuit image (upload one via the file browser) and re-run this cell.


## 12. Notes & troubleshooting

- **`could not make a flat list of images`** → ensure `to_conversation` passes PIL images (it does) and exactly one image per sample.
- **`max_length` vs `max_seq_length`** → newer TRL uses `max_length` (used here). If your TRL errors, rename it to `max_seq_length`.
- **OOM mid-run** → lower `MAX_IMAGE_PX` (768→512), keep `use_gradient_checkpointing="unsloth"`, `BATCH=1`, raise `GRAD_ACCUM`; last resort, `finetune_vision_layers=False` or a 2B model.
- **Tuned barely beats base** → concentrate the eval on harder cases (more components, parallel blocks) and on the abstention subset, where the base fails most; check the loss actually dropped; add render variety in the data.
- **Scaling** → build data for 50k but train up from ~10-15k, watching the section-9 curve; stop when it flattens.


## 13. (v2) DPO preference tuning — build this AFTER v1 succeeds

> **Do not build/run this yet.** DPO is a *second stage that runs on top of the v1 fine-tuned model*, not an alternative to it. It only becomes meaningful — and measurable — once v1 has posted a solid base-vs-tuned gain. Running it on a weak v1 model just adds noise you can't interpret.

**When to come back here:** after section 9 shows the tuned model clearly beating the base on component / R_eq / answer accuracy (and ideally on the real-world set in section 10).

**What it will do when built:**
- Load `train_pairs.jsonl` (already produced by the data notebook when `INCLUDE_MISTAKES=True`).
- Form preference pairs: `correct_solution` = *chosen*, `wrong_solution` = *rejected*, same image + question as the prompt.
- Run DPO on top of the v1 LoRA adapter (Unsloth supports vision DPO), which sharpens the model *away* from the exact mistakes in the pairs (parallel-as-series, omitted branch, Ohm's-law flip, misread value).
- Re-run the section-9 harness to check DPO improved spec adherence *beyond* SFT alone — especially the fabrication / setup-error cases.

**Why it's deferred, not written now:** vision DPO has its own trainer, a reference-model copy (tighter VRAM than SFT), and settings that depend on what v1 actually produced and which model/hardware the Day-1 smoke test landed on. Writing it before v1 exists risks writing it twice. The data is already waiting, so nothing is blocked by deferring.